
# RSNA Knee — Final Clean 0.891 Baseline + Enhanced Rank Ensemble

This notebook consolidates the useful parts of the three supplied notebooks:

- the **clean EDA / metadata audit** from the older notebook;
- the **proven 20-checkpoint DINOv2-small MRI pipeline** from `rsna-knee-baseline-v1`;
- the stronger report-analysis notebook's key conclusion that its scored MRI path is the same 20-member ensemble.

The original model/data conventions are preserved. The final change is deliberately at the **ensemble layer**, where it cannot silently change the pixels that the imported checkpoints were trained to read.

It writes the exact public-style baseline plus three experimental AUC-oriented alternatives. `submission.csv` defaults to the conservative hybrid; set `RSNA_ENSEMBLE_MODE=baseline` to reproduce the proven equal-rank branch.


## Before running on Kaggle

Keep the same inputs as the supplied 0.891 notebook:

- Competition data: `/kaggle/input/competitions/rsna-knee-abnormality-detection` (the fallback `/kaggle/input/rsna-knee-abnormality-detection` is also supported).
- Ensemble weights: `/kaggle/input/rsna-knee-weights` containing `manifest.json` and its 20 member checkpoints.
- DINOv2-small base model: attach the same Kaggle model/dataset used by the source notebook. The code discovers the mounted DINOv2 directory by its `config.json`, so no new hard-coded model path is introduced.

No dataset or checkpoint file is copied into this notebook; you can attach them manually exactly as before.

## 1. Configuration — keep the original dataset/model mounts

In [ ]:
from __future__ import annotations

import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import hashlib
import json
import re
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display

T0 = time.time()
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

# -----------------------------------------------------------------------------
# IMPORTANT: these are the same competition/model conventions used by the
# 0.891 public pipeline. The manifest will overwrite pixel settings member by
# member before decoding, so do not "improve" crop/order/laterality independently
# of the checkpoint that was trained on them.
# -----------------------------------------------------------------------------
PREFERRED_WEIGHTS = Path("/kaggle/input/rsna-knee-weights")
CROP_MM = 130.0
CACHE_IMG = 336
IMG = CACHE_IMG
GROUP = 3
CACHE_SLICES = 12
N_GROUP = max(CACHE_SLICES // GROUP, 1)
SLICE_BAND = (0.20, 0.80)

HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600
LAT_MIN_OFFSET_MM = 20.0
LEGACY_LAT_OFFSET_MM = 5.0

RULES_NATIVE = {
    "order": "normal",
    "lat": "centre",
    "slot_fallback": False,
    "decode_fill": "nearest",
}
RULES_LEGACY = {
    "order": "dominant_axis",
    "lat": "corner_x",
    "slot_fallback": True,
    "decode_fill": "zero",
}
RULES = dict(RULES_NATIVE)

SLOTS_RECOVERED = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]
SLOTS_PUBLIC = [
    ("SAG_FLUID", "Sagittal", None, True),
    ("COR_FLUID", "Coronal", None, True),
    ("AX_FLUID", "Axial", None, True),
    ("SAG_STRUCT", "Sagittal", None, False),
    ("COR_STRUCT", "Coronal", None, False),
    ("AX_STRUCT", "Axial", None, False),
]
SLOT_SCHEME = os.environ.get("SLOT_SCHEME", "recovered")
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == "public" else SLOTS_RECOVERED
N_SLOT = len(SLOTS)

POOL_PARTS = {"cls_mean": 2, "cls_mean_focal": 3}
SLOT_PRIOR_TABLE = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}
SLOT_PRIOR_STRENGTH = 0.55

FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(
    r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
    r"water excit|\btirm\b|\bsting\b|\bfatsup\b"
)
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")

# AUC-oriented ensemble controls. Baseline is always written separately.
ENSEMBLE_MODE = os.environ.get("RSNA_ENSEMBLE_MODE", "hybrid").lower()
QUALITY_TEMPERATURE = 0.35
QUALITY_SHRINK = 0.35
HYBRID_BASE = 0.75
HYBRID_QUALITY = 0.20
HYBRID_WINDOW = 0.05
HYBRID_TARGET_BASE = 0.55
HYBRID_TARGET_QUALITY = 0.35
HYBRID_TARGET_WINDOW = 0.10

# The old ~0.83 checkpoint is intentionally NOT mixed by default. A known weaker,
# differently preprocessed model is useful only after OOF complementarity is shown.


In [ ]:
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


def find_root():
    for c in [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("data"),
        Path("."),
    ]:
        if (c / "test.csv").is_file() and (c / "test_series").is_dir():
            return c
    base = Path("/kaggle/input")
    if base.is_dir():
        for depth1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for cand in [depth1] + sorted(p for p in depth1.iterdir() if p.is_dir()):
                if (cand / "test.csv").is_file() and (cand / "test_series").is_dir():
                    return cand
    raise FileNotFoundError("RSNA knee competition mount not found")


def find_dinov2(variant="small"):
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if "config.json" in files and "dinov2" in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None


ROOT = find_root()
log(f"input root: {ROOT}")


## 2. Lightweight EDA and annotation sanity checks

In [ ]:
def read_csv_if_present(name):
    p = ROOT / name
    if not p.is_file():
        return pd.DataFrame()
    return pd.read_csv(p)


train_df_eda = read_csv_if_present("train.csv")
train_series_eda = read_csv_if_present("train_series.csv")
test_df_eda = read_csv_if_present("test.csv")
test_series_eda = read_csv_if_present("test_series.csv")

summary = []
for name, frame in [
    ("train.csv", train_df_eda),
    ("train_series.csv", train_series_eda),
    ("test.csv", test_df_eda),
    ("test_series.csv", test_series_eda),
]:
    if not frame.empty:
        summary.append({"file": name, "rows": len(frame), "columns": len(frame.columns)})
display(pd.DataFrame(summary))

# Correct prevalence: missing labels are unknown, NOT negative.
present_targets = [t for t in TARGETS if t in train_df_eda.columns]
if present_targets:
    n_labeled = train_df_eda[present_targets].notna().sum()
    positives = train_df_eda[present_targets].sum(skipna=True)
    negatives = n_labeled - positives
    stats = pd.DataFrame({
        "annotated": n_labeled,
        "positive": positives,
        "negative": negatives,
        "positive_rate_on_annotated": positives / n_labeled.replace(0, np.nan),
    })
    display(stats.style.format({"positive_rate_on_annotated": "{:.1%}"}))
    fully = int(train_df_eda[present_targets].notna().all(axis=1).sum())
    print(f"Studies with all 12 image annotations: {fully} / {len(train_df_eda)}")

if not train_series_eda.empty:
    cols = [c for c in ["Fluid_Sensitive", "Fat_Suppression", "Anatomical_Plane"]
            if c in train_series_eda.columns]
    if cols:
        display(train_series_eda.groupby(cols, dropna=False).size()
                .rename("series_count").reset_index()
                .sort_values("series_count", ascending=False).head(30))
    per_study = train_series_eda.groupby("StudyInstanceUID").size()
    display(per_study.describe().rename("series_per_study").to_frame())



**Important:** the competition CSV sequence flags are audited, not forced into the imported 0.891 checkpoints. Their manifest records the exact recovered sequence/preprocessing contract used during training; changing that only at inference would be a train/test preprocessing mismatch.


## 3. DICOM acquisition metadata and recovered sequence semantics

In [ ]:
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing", "Rows",
            "Columns", "RescaleSlope", "RescaleIntercept",
            # Position and orientation are read from the same header probe() already
            # opens, so they cost nothing, and they are what recovers the side when the
            # Laterality tag is absent - which it is for half the studies here.
            "ImagePositionPatient", "ImageOrientationPatient"]


def _hdr_vec(s, n):
    """Parse a DICOM multi-value string as stored by probe(): floats joined by `|`."""
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split("|")]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None


def side_from_geometry(h):
    """Study -> 'L' / 'R' / None, from where the image sits in the patient.

    `Laterality` (0020,0060) is Type 2C and may legitimately be absent; in this corpus it
    is missing on exactly half the studies, and the vendors it is missing from are whole
    vendors rather than scattered series. A study with no tag is not a left knee, but the
    normalisation upstream treats it as one, so half the corpus was never normalised and
    the five side-defined targets - the two menisci, the two tibiofemoral compartments
    and the medial collateral ligament - saw that axis reversed on a large minority of it.

    The patient coordinate system fixes this without the tag: +x is the patient's left, so
    the centre of a right knee sits at negative x. The centre is used rather than
    `ImagePositionPatient` itself because that is the corner of the image, which is offset
    by half a field of view - enough to change the sign on a knee near the midline.

    The median over a study's series is what is thresholded, not a single series: probe()
    reads one arbitrary slice per series, which on a sagittal stack can sit anywhere
    across the joint. Studies whose centre falls near the midline are left unresolved
    rather than guessed - measured against the tagged half, the rule is right 97% of the
    time overall and no better than chance inside 20 mm.
    """
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
        iop = _hdr_vec(getattr(r, "ImageOrientationPatient", None), 6)
        ps = _hdr_vec(getattr(r, "PixelSpacing", None), 2)
        rows, cols = getattr(r, "Rows", None), getattr(r, "Columns", None)
        if ipp is None or iop is None or ps is None or not rows or not cols:
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else ("R" if m < 0 else "L")
    return out


def side_from_corner_x(h):
    """The laterality an imported member was fitted under.

    It thresholds the median raw `ImagePositionPatient` x over a study's series. That is
    the x of the image *corner*, not of its centre, so it differs from the rule above by
    up to half a field of view - which is enough to reverse the sign on a knee scanned
    near the midline. The dead zone is 5 mm rather than 20 mm, so it also commits on
    studies the rule above leaves unresolved.

    Neither difference changes a shape. Each one decides whether a study is mirrored, and
    a study mirrored one way at training and the other at inference presents the five
    side-defined targets with their axis reversed.
    """
    out = {}
    for st, g in h.groupby("StudyInstanceUID"):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        # DICOM patient coordinates are LPS: +x is the patient's left.
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else ("R" if x < 0 else "L")
    return out


def lat_of(h, tag=""):
    """Study -> 'L' / 'R' / None: the tag where it exists, geometry where it does not.

    The tag is present on exactly half the studies here and is sometimes an empty
    string rather than absent, which is not the same as NaN. Treating the other half
    as left-sided is what `normalise_laterality` did by omission, so the geometry
    fallback is not a refinement - it is the difference between normalising half the
    corpus and normalising all of it.
    """
    geo = side_from_corner_x(h) if RULES["lat"] == "corner_x" else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = {}, 0, 0, 0, 0
    for st, g in h.groupby("StudyInstanceUID"):
        v = [str(x).strip().upper() for x in g["Laterality"].dropna()]
        if RULES["lat"] == "corner_x" and "ImageLaterality" in g.columns:
            # The legacy rule reads the second tag too, so a study tagged only there is
            # resolved from the tag rather than from geometry.
            v += [str(x).strip().upper() for x in g["ImageLaterality"].dropna()]
        v = [x[0] for x in v if x and x[0] in ("L", "R")]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f"{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, "
        f"{n_none} unresolved; tag and geometry disagree on {n_disagree} "
        f"({n_disagree / max(n_tag, 1):.1%} of the tagged)")
    return d



def probe(item):
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
           "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def walk(split):
    """Every series directory of a split, with one header read per series.

    An absent split returns an empty frame *with the columns annotate expects*. Returning
    a bare DataFrame looks like the same thing and is not: the next call indexes
    `SeriesDescription` and raises KeyError, so the branch that exists to survive a
    missing split is what turns it into a crash.
    """
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=["split", "StudyInstanceUID", "SeriesInstanceUID",
                                     "dir", "files", "n_slices"] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)


def annotate(df):
    """Recover fat suppression and pulse-sequence weighting from the header."""
    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    # GE writes SAT_GEMS for spatial saturation, so ScanOptions must be matched as
    # exact tokens; a substring test on "SAT" fires on non-fat-sat series.
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)

    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df


In [ ]:
def _as_bool(v):
    if pd.isna(v):
        return None
    s = str(v).strip().upper()
    if s in {"1", "TRUE", "T", "YES", "Y"}:
        return True
    if s in {"0", "FALSE", "F", "NO", "N"}:
        return False
    try:
        f = float(s)
        return True if f == 1 else False if f == 0 else None
    except Exception:
        return None


def audit_official_sequence_metadata(inferred, official):
    """Audit only. Do not change imported checkpoint pixels from this table."""
    need = {"SeriesInstanceUID", "Fluid_Sensitive", "Fat_Suppression"}
    if inferred.empty or official.empty or not need.issubset(official.columns):
        return
    a = inferred[["SeriesInstanceUID", "fluid", "fatsat"]].copy()
    b = official[["SeriesInstanceUID", "Fluid_Sensitive", "Fat_Suppression"]].copy()
    b["official_fluid"] = b["Fluid_Sensitive"].map(_as_bool)
    b["official_fatsat"] = b["Fat_Suppression"].map(_as_bool)
    m = a.merge(b[["SeriesInstanceUID", "official_fluid", "official_fatsat"]],
                on="SeriesInstanceUID", how="inner")
    for inferred_col, official_col, name in [
        ("fluid", "official_fluid", "Fluid_Sensitive"),
        ("fatsat", "official_fatsat", "Fat_Suppression"),
    ]:
        valid = m[official_col].notna() & m[inferred_col].notna()
        if valid.any():
            agree = (m.loc[valid, inferred_col].astype(bool).values ==
                     m.loc[valid, official_col].astype(bool).values).mean()
            log(f"metadata audit {name}: {agree:.1%} agreement on {int(valid.sum())} series")


## 4. Select the six MRI slots exactly as the checkpoint expects

In [ ]:
def pick_slots(series_df, plane_map):
    """One series per slot per study.

    Ties are broken toward the stack with the most slices: a thicker stack samples the
    joint more densely, and the three-slice sampler below benefits from the margin.
    """
    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs)
            # fluid=None means "do not condition on weighting" - the public scheme,
            # where the single provided flag stands in for both axes at once.
            if fluid is not None:
                sel &= (g["fluid"] == fluid)
            cand = g[sel]
            # A slot with no series matching its predicate stays empty, and no substitute
            # is admitted from a neighbouring predicate. Relaxing the weighting to fill a
            # T1 slot would draw from the pool `SAG_FLUID_NOFS` selects from, since that
            # pool is what remains once the weighting is dropped: over the training corpus
            # it would put one series in two slots for 2383 of 4407 studies and leave 56%
            # of the T1 slot holding PD or T2. The presence mask would then assert a
            # sequence that was never acquired, and the per-diagnosis softmax of §6 would
            # divide its attention across two identical slots, giving one acquisition
            # about twice the weight it carries in a study that holds both. The mask is
            # there to say a slot is absent, which is what an absent slot is.
            if len(cand) == 0 and RULES["slot_fallback"] and fluid is False:
                # The relaxation the paragraph above rejects, reproduced because an
                # imported member was fitted with its T1 slots filled this way: over half
                # of that member's training studies had a T1 slot holding a series that
                # is not T1. Leaving those slots empty would present it with a presence
                # mask it never saw.
                cand = g[(g["plane"] == plane) & (~g["fatsat"])]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out


## 5. Physical slice ordering, crop, intensity normalisation

In [ ]:
ORDER_TAGS = [(0x0020, 0x0032), (0x0020, 0x0037), (0x0020, 0x0013)]

# Series in which at least one sampled slice would not decode. A list rather than a
# counter because appending is atomic under the reader threads, and reported rather than
# swallowed: unreported, a decode failure is indistinguishable from a black knee.
DECODE_FAILED = []


def cache_tag(rules=None):
    """The name a decoded cache is stored under.

    It has to name everything that decides the pixels, not only their dimensions. Two
    configurations that agree on resolution, slice count, crop and band but disagree on
    how a slice is chosen produce different arrays of identical shape - so a tag built
    from the dimensions alone lets the second attach to the first one's file and train
    against pixels it never asked for, with nothing anywhere reporting a mismatch.

    A native reading keeps the plain name, so caches decoded before the rules existed
    stay valid; anything else earns a suffix.
    """
    r = dict(RULES if rules is None else rules)
    t = (f"{CACHE_IMG}px_{CACHE_SLICES}sl_{int(CROP_MM)}mm_"
         f"{SLICE_BAND[0]:.2f}-{SLICE_BAND[1]:.2f}")
    if {k: r.get(k, v) for k, v in RULES_NATIVE.items()} != RULES_NATIVE:
        t += "_" + hashlib.md5(json.dumps(r, sort_keys=True).encode()).hexdigest()[:6]
    return t


def _natural_key(name):
    return tuple(int(x) if x.isdigit() else x.lower()
                 for x in re.split(r"(\d+)", str(name)))


def _order_dominant_axis(rec):
    """The slice order an imported member was fitted under.

    It sorts on the raw patient coordinate along whichever axis varies most across the
    stack, rather than on the projection onto the slice normal. The two differ by a sign,
    not by a formula: measured over this corpus every sagittal series has a slice normal
    with n_x in [-1.00, -0.98], so p.n is the negative of the raw x this sorts on and the
    two stacks come out exactly reversed. Because the band sampler truncates rather than
    rounds, its nine indices are not symmetric about the middle, so nine slices drawn from
    a twenty-six slice stack under one order share two with the other.

    Missing geometry falls back to `InstanceNumber` and then to a natural sort of the file
    name, both at the same 80% threshold the imported pipeline used.
    """
    files, d = rec["files"], rec["dir"]
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=["ImagePositionPatient", "InstanceNumber"])
            raw = getattr(ds, "ImagePositionPatient", None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, "InstanceNumber", None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))

    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare,
                                 r[2] if r[2] is not None else float("inf"), r[3]))
    elif sum(r[2] is not None for r in rows) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float("inf"), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return [r[0] for r in rows], True


def order_slices(rec):
    """Return the series' files sorted along the through-plane axis.

    A DICOM file name here is a SOP Instance UID, which is assigned arbitrarily. Sorting
    by it therefore produces an order uncorrelated with anatomy - measured over one
    series, Spearman between file-name rank and physical position is 0.009, i.e. none.
    Anything that assumes the file order means something is then operating on noise: the
    three channels of a "2.5D" input are three unrelated views rather than neighbouring
    slices, "the middle of the stack" is a random subset, and reversing slice order to
    normalise laterality reverses nothing meaningful.

    The physical order is recoverable exactly. Each slice carries its position in patient
    coordinates and the in-plane axes; projecting the position onto the slice normal
    gives a signed through-plane coordinate, monotonic along the stack:

        n = r_x  x  r_y ,      k = p . n

    `InstanceNumber` is the fallback. It usually tracks the projection up to sign, but
    interleaved and multi-echo acquisitions need not number slices in the order they
    occupy in space - but the projection is signed in patient
    coordinates, which is what laterality normalisation needs.
    """
    if RULES["order"] == "dominant_axis":
        return _order_dominant_axis(rec)
    files, d = rec["files"], rec["dir"]
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any(k is None for k, _ in keyed):
        # A series with no usable geometry keeps its arbitrary order; that is worse than
        # sorting but better than dropping the series, and it is logged as a count.
        return files, False
    return [f for _, f in sorted(keyed, key=lambda t: t[0])], True


def read_slot(rec, n_slice=None, out_size=None):
    """`n_slice` physically spread slices from one series, at `out_size` pixels.

    Returns uint8 [n_slice, out, out] normalised per-series to its 1st-99th
    percentile. Percentiles rather than min/max because MR intensity has no absolute
    scale and a single bright vessel would otherwise compress the whole dynamic range.

    Reading is the expensive half of this pipeline, so the caller reads once at the
    largest configuration it needs and derives the smaller ones from the returned buffer
    rather than re-reading.
    """
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = rec.get("ordered") or rec["files"], rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None
    # Spread the samples over a central band of the stack: the outermost slices of a knee
    # series are mostly soft tissue outside the joint. The band is a constant rather than
    # a literal because how much of the stack is worth reading depends on how many slices
    # are being taken - at three the middle is all that fits, while at sixteen the ends
    # are worth having, and a Baker cyst sits at the posteromedial end of a sagittal one.
    lo, hi = int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])

    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None                      # no shape is known here; see below
        planes.append(a)

    # A slice that would not decode has no shape of its own, and inventing one is how a
    # single unreadable file erases a whole series: a substitute allocated at the resize
    # target while the decoded slices are still native makes the shape check below take
    # the substitute as the authority and zero the good slices with it, leaving a black
    # slot that the presence mask still reports as acquired.
    #
    # A failure is instead filled from the nearest slice that did decode - the same
    # convention the sampler already uses when the band holds fewer distinct slices than
    # were asked for - and a series where nothing decodes is reported absent, which the
    # mask can express, rather than black, which it cannot.
    got = [k for k, p in enumerate(planes) if p is not None]
    if RULES["decode_fill"] == "zero":
        # What an imported member was fitted with: a failure becomes a zero plane at the
        # resize target, which the shape check below then propagates to the whole slot.
        # It is the behaviour the paragraph above describes and rejects, kept here only
        # because that member's weights were learned against slots blacked out this way.
        if not got:
            DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p
                  for p in planes]
        got = list(range(len(planes)))
    if not got:
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]

    # Slices of one series can still differ in matrix size - multi-echo and some
    # reformats do - and those are genuinely not stackable.
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)

    # constant physical extent, then resize: PixelSpacing varies 3.4x across the corpus
    if px and np.isfinite(px) and px > 0:
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    # uint8, not float32. These buffers queue up between the reader threads and the
    # encoder, and at this size a float32 slot-series is several megabytes. Intensity is
    # already normalised into [0, 1] here, so eight bits cost nothing that a bilinear
    # resize has not already cost, and the queue is a quarter the size.
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


## 6. Laterality normalisation

In [ ]:
def normalise_laterality(img, plane, lat):
    """Map every knee onto a left-knee convention.

    Coronal and axial views mirror under a horizontal flip. Sagittal stacks are not
    mirror images of each other - the slice order runs medial-to-lateral in opposite
    directions - so the channel order is reversed instead.
    """
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])


## 7. Decode each selected series once into the study cache

In [ ]:
# Where the geometric slice order may be remembered between runs. Unset on the platform,
# because each run gets a fresh machine and there is nothing to remember; set off it,
# where the same corpus is cached again at every resolution and slice count and the order
# is a function of neither. It is opt-in so that the scored run's behaviour is decided by
# the code rather than by whether a file happens to be lying about.
ORDER_CACHE = os.environ.get("RSNA_ORDER_CACHE") or None


def build_cache(slot_map, plane_map, lat_map, tag):
    """Decode every (study, slot) once into an in-memory uint8 array.

    Fine-tuning revisits the same pixels every epoch. Reading them from the mount each
    time would make the epoch count a function of I/O rather than of learning, so they
    are decoded once and held as bytes: intensity has already been normalised into
    [0, 1], and eight bits cost nothing a bilinear resize has not already cost.

    CACHE_SLICES positions are kept per slot, which the training loop reads as N_GROUP
    groups of GROUP consecutive channels.
    """
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    n_job = len(jobs)

    # Ordering first, and as its own pass. It reads one header per slice of every chosen
    # series - far more file opens than the decode that follows - and on a network mount
    # that is latency, not work, so it gets its own wider pool.
    t_ord = time.time()
    n_slice_total = sum(len(j[3]["files"]) for j in jobs)
    log(f"{tag}: ordering {len(jobs)} slot-series ({n_slice_total} slice headers)")
    ok = done = 0
    CHUNK_O = 1024

    # A remembered order, when one is offered. The projection depends on the DICOM
    # geometry alone, so it is the same at every resolution and every slice count, and
    # it costs one header read per slice - the largest single cost in this pass. An entry
    # is validated by the number of files present, so a tree that has changed under it is
    # recomputed rather than trusted: order is derived data, and a stale entry would be
    # invisible in the way that matters most.
    seen = {}
    if ORDER_CACHE and Path(ORDER_CACHE).is_file():
        try:
            import json as _json
            seen = _json.loads(Path(ORDER_CACHE).read_text())
        except (OSError, ValueError):
            seen = {}
        hit = 0
        for _, _, _, rec in jobs:
            e = seen.get(rec["SeriesInstanceUID"])
            if e and len(e["files"]) == len(rec["files"]):
                rec["ordered"] = e["files"]
                ok += int(e["good"])
                hit += 1
        jobs = [j for j in jobs if "ordered" not in j[3]]
        log(f"{tag}: {hit} slot-series ordered from {ORDER_CACHE}, {len(jobs)} to read")

    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK_O):
            block = jobs[c0:c0 + CHUNK_O]
            for (_, _, _, rec), (files, good) in zip(
                    block, pool.map(lambda j: order_slices(j[3]), block)):
                rec["ordered"] = files
                ok += int(good)
                done += 1
                if ORDER_CACHE:
                    seen[rec["SeriesInstanceUID"]] = {"files": files, "good": bool(good)}
            # The ceiling is whichever comes first: the pass's own budget, or the share
            # of what is left of the run that it may take. The second is what makes the
            # first safe to set generously - a mount slow enough to matter cannot spend
            # the training time, because the budget shrinks as the run does.
            budget = min(ORDER_BUDGET_S, max(60.0, (TIME_BUDGET - (time.time() - T0)) * 0.35))
            if time.time() - t_ord > budget:
                log(f"{tag}: ordering budget spent at {done}/{len(jobs)}; "
                    f"the rest keep file order")
                break
    if ORDER_CACHE and done:
        import json as _json
        _t = Path(ORDER_CACHE).with_suffix(".tmp")
        _t.write_text(_json.dumps(seen))
        _t.replace(Path(ORDER_CACHE))
    log(f"{tag}: ordered {ok}/{n_job} by geometry "
        f"({n_job - ok} kept arbitrary) in {time.time() - t_ord:.0f}s")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    log(f"{tag}: decoding {len(jobs)} slot-series")
    n_failed_before = len(DECODE_FAILED)

    CHUNK = 512
    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            block = jobs[c0:c0 + CHUNK]
            for (st, k, plane, _), img in zip(
                    block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane,
                                                          lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < CHUNK:
                log(f"  {tag} {done}/{len(jobs)}")
            if time.time() - T0 > TIME_BUDGET:
                log(f"  {tag}: time budget reached during decode")
                break
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f"{tag}: {int(mask.sum())}/{len(jobs)} slots filled"
        + (f"; {n_failed} series had a slice that would not decode" if n_failed else ""))
    gc.collect()
    return studies, cache, mask


## 8. Checkpoint-compatible DINOv2 multi-slot model

In [ ]:
class SlotHead(nn.Module):
    """Per-diagnosis attention over the slot embeddings of one study.

    Each finding is read on particular sequences - cruciates sagittally, collateral
    ligaments and the meniscal body coronally, patellar cartilage axially - so pooling
    the slots identically would dilute the one that carries the evidence with the rest.

    The aggregation is deliberately this simple. With a study-level label there is no
    signal telling the model which part of a study matters, so extra attention
    parameters below the slot level would have nothing to learn from and would spend
    their capacity fitting noise.
    """

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        # An imported member carries a fixed per-(diagnosis, slot) tilt on the attention
        # logits, set from the anatomy table below rather than learned. It is a buffer, so
        # it travels in the state dict and must exist for that member to load; exp(0.55)
        # gives a preferred slot about 1.73x the weight of an unpreferred one, which
        # biases the softmax without ever excluding a slot.
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and n_out == len(TARGETS):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer("slot_prior", p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


In [ ]:
class Model(nn.Module):
    """Encoder plus head, trained end to end.

    A study arrives as a bag of slot images. The bag is flattened for the encoder and
    folded back before the head, so the encoder never sees the study structure and the
    head never sees pixels.
    """

    def __init__(self, backbone, dim, pool="cls_mean", prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            # The cache is held at the highest resolution any configuration needs; the
            # rest downsample from it, so every configuration sees the same pixels
            # through a different sampling grid rather than a different crop.
            x = F.interpolate(x, size=(img_size, img_size), mode="bilinear",
                              align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == "cls_mean_focal":
            # The upper tail of each channel over the patch grid, taken per channel
            # rather than by selecting whole patches: a finding occupies a small part of
            # the field, so a plain mean over 256 patches dilutes it by two orders of
            # magnitude, and this keeps the top eighth of each channel's responses.
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)


In [ ]:
def build_model(unfreeze_last, source=None, variant="small", pool="cls_mean",
                prior=False):
    """Load the encoder and open the last `unfreeze_last` blocks for training.

    The early blocks of a self-supervised transformer are generic edge and texture
    filters; the late blocks carry semantics. Opening only the late ones is the cautious
    choice - there may not be enough supervision here to improve the early ones and there
    is certainly enough to damage them - but how far the line should sit is a question
    the corpus has to answer rather than the intuition.

    `source` names where the weights come from. Left unset it is the attached model
    directory, which is the only thing available here. It is a parameter so that a run
    off the platform builds the same object from the same code rather than from a second
    definition that has to be kept in step by hand.
    """
    from transformers import AutoModel
    p = source if source is not None else find_dinov2(variant)
    if p is None:
        raise FileNotFoundError("DINOv2 weights not attached")
    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    dim = bb.config.hidden_size
    trainable = sum(p.numel() for p in bb.parameters() if p.requires_grad)
    log(f"backbone: {n_layer} blocks, last {unfreeze_last} trainable "
        f"({trainable / 1e6:.1f}M params), feature dim {dim * POOL_PARTS[pool]}")
    return Model(bb, dim, pool=pool, prior=prior)



## 9. Enhanced AUC-oriented ensemble

The supplied 0.891 notebook averages per-member ranks equally. This final version **keeps that exact branch** and adds a conservative alternative:

1. Within each fold, configurations are softly weighted by their holdout score. Different folds are never compared by raw holdout value; every fold receives equal total mass.
2. Those weights are heavily shrunk toward uniform weights to avoid overfitting a small validation difference.
3. TTA windows contribute a small rank-space vote, aligning the aggregation with the mean-AUC metric.
4. If the attached package contains per-target validation AUCs, they are used target-by-target; otherwise the scalar holdout is used.

No score above 0.891 can be guaranteed from inference-only changes on an unseen leaderboard, so the proven baseline is always saved beside the hybrid.


In [ ]:
# -----------------------------------------------------------------------------
# Enhanced inference: the exact 0.891 rank-mean remains available as a safety file.
# Improvements are conservative and AUC-oriented:
#   1) compare configurations only WITHIN the same fold, then equalise fold mass;
#   2) shrink quality weights strongly toward equal weighting;
#   3) give each TTA window a small rank vote instead of trusting calibration alone;
#   4) use per-target validation AUC only if the attached manifest/checkpoint actually
#      contains it; otherwise fall back to scalar holdout quality.
# -----------------------------------------------------------------------------

FINGERPRINT_TOL = 2e-3
TTA_OVERLAP = True
TTA_POOL = "prob"


def fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    n_slot = N_SLOT if n_slot is None else n_slot
    group = GROUP if group is None else group
    seed = SEED if seed is None else seed
    g = torch.Generator().manual_seed(seed)
    imgs = torch.randint(0, 256, (2, n_slot, group, img_size, img_size),
                         generator=g, dtype=torch.uint8).to(dev)
    mask = torch.ones(2, n_slot, device=dev)
    mask[1, -1] = 0.0
    was_training = model.training
    model.eval()
    with torch.no_grad():
        out = model(imgs, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out


class WeightsError(RuntimeError):
    pass


def check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=""):
    got = fingerprint(model, dev, img_size)
    exp = np.asarray(expected, np.float32)
    if got.shape != exp.shape:
        raise WeightsError(f"{tag}fingerprint shape {got.shape} != stored {exp.shape}")
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(
            f"{tag}fingerprint differs by {d:.4g} > {tol:g}; architecture/preprocessing contract moved")
    log(f"{tag}fingerprint matches within {d:.2g}")
    return d


def _valid_package(root, name="manifest.json"):
    root = Path(root)
    p = root / name
    if not p.is_file():
        return False
    try:
        man = json.loads(p.read_text())
    except Exception:
        return False
    members = man.get("members")
    if not isinstance(members, list) or not members:
        return False
    missing = [m.get("file") for m in members if not (root / str(m.get("file"))).is_file()]
    if missing:
        raise WeightsError(f"weights package {root} is incomplete; first missing: {missing[0]}")
    return True


def find_weights(name="manifest.json"):
    # Keep the original model path first, exactly as requested.
    if _valid_package(PREFERRED_WEIGHTS, name):
        return PREFERRED_WEIGHTS
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if name in files and _valid_package(Path(root), name):
            return Path(root)
    return None


def window_starts(n_slice, group, overlap=None):
    overlap = TTA_OVERLAP if overlap is None else overlap
    if overlap and n_slice >= group:
        return list(range(n_slice - group + 1))
    return [g * group for g in range(max(n_slice // group, 1))]


@torch.no_grad()
def predict_member_views(model, cache, mask, idx, dev, img_size, group=None,
                         pool=None, starts=None):
    """Return original member prediction AND every TTA-window probability."""
    group = GROUP if group is None else group
    pool = TTA_POOL if pool is None else pool
    starts = window_starts(cache.shape[2], group) if starts is None else list(starts)
    if not starts:
        raise ValueError("no TTA windows")

    model.eval()
    mean_pred = np.empty((len(idx), len(TARGETS)), np.float32)
    view_pred = [np.empty_like(mean_pred) for _ in starts]

    for b0 in range(0, len(idx), EVAL_BATCH):
        sel = idx[b0:b0 + EVAL_BATCH]
        b1 = b0 + len(sel)
        m = torch.from_numpy(mask[sel]).to(dev)
        acc = None
        for wi, st in enumerate(starts):
            rows = torch.from_numpy(
                np.ascontiguousarray(cache[sel, :, st:st + group])).to(dev)
            with torch.autocast("cuda", enabled=dev.type == "cuda"):
                z = model(rows, m, img_size).float()
            prob = torch.sigmoid(z)
            view_pred[wi][b0:b1] = prob.cpu().numpy()
            v = z if pool == "logit" else prob
            acc = v if acc is None else acc + v
        v = acc / len(starts)
        if pool == "logit":
            v = torch.sigmoid(v)
        mean_pred[b0:b1] = v.cpu().numpy()
    return mean_pred, view_pred


def _rank_matrix(x):
    return pd.DataFrame(np.asarray(x)).rank(method="average", pct=True).to_numpy(np.float64)


def _parse_auc_vector(obj):
    if obj is None:
        return None
    if isinstance(obj, dict):
        if all(t in obj for t in TARGETS):
            try:
                a = np.array([float(obj[t]) for t in TARGETS], dtype=np.float64)
                return a if np.isfinite(a).any() else None
            except Exception:
                return None
    if isinstance(obj, (list, tuple, np.ndarray)) and len(obj) == len(TARGETS):
        try:
            a = np.asarray(obj, dtype=np.float64)
            return a if np.isfinite(a).any() else None
        except Exception:
            return None
    return None


def extract_target_auc(member, checkpoint):
    keys = ["per_target_auc", "target_auc", "auc_by_target", "label_auc", "val_auc_by_target"]
    for container in (member, checkpoint):
        if not isinstance(container, dict):
            continue
        for k in keys:
            a = _parse_auc_vector(container.get(k))
            if a is not None:
                return a
        metrics = container.get("metrics")
        if isinstance(metrics, dict):
            for k in keys:
                a = _parse_auc_vector(metrics.get(k))
                if a is not None:
                    return a
    return None


def _shrunk_softmax(scores):
    scores = np.asarray(scores, dtype=np.float64)
    n = len(scores)
    if n == 1:
        return np.ones(1, dtype=np.float64)
    valid = np.isfinite(scores)
    if valid.sum() < 2:
        return np.full(n, 1.0 / n)
    fill = np.nanmedian(scores[valid])
    s = np.where(valid, scores, fill)
    sd = float(np.std(s))
    if sd < 1e-8:
        return np.full(n, 1.0 / n)
    z = np.clip((s - s.mean()) / sd, -2.0, 2.0)
    e = np.exp(QUALITY_TEMPERATURE * z)
    soft = e / e.sum()
    uniform = np.full(n, 1.0 / n)
    return (1.0 - QUALITY_SHRINK) * uniform + QUALITY_SHRINK * soft


def fold_balanced_member_weights(per_member, target_specific=False):
    """Weights sum to 1 per label; every fold gets equal total mass.

    Raw holdout scores from DIFFERENT folds are never compared. Within one fold the
    candidate configurations saw the same held-out studies, so relative quality is usable.
    """
    m = len(per_member)
    W = np.zeros((m, len(TARGETS)), dtype=np.float64)
    folds = sorted({str(x.get("fold")) for x in per_member})
    fold_mass = 1.0 / max(len(folds), 1)

    for f in folds:
        ids = [i for i, x in enumerate(per_member) if str(x.get("fold")) == f]
        scalar = np.array([
            float(per_member[i]["holdout"]) if per_member[i].get("holdout") is not None else np.nan
            for i in ids
        ], dtype=np.float64)
        scalar_w = _shrunk_softmax(scalar)

        for t in range(len(TARGETS)):
            if target_specific:
                vals = []
                for i in ids:
                    a = per_member[i].get("target_auc")
                    vals.append(a[t] if a is not None and np.isfinite(a[t]) else np.nan)
                vals = np.asarray(vals, dtype=np.float64)
                w = _shrunk_softmax(vals) if np.isfinite(vals).sum() >= 2 else scalar_w
            else:
                w = scalar_w
            W[ids, t] = fold_mass * w

    # Numerical guard; also covers a malformed manifest with no fold values.
    den = W.sum(axis=0, keepdims=True)
    bad = den[0] <= 0
    if bad.any():
        W[:, bad] = 1.0 / max(m, 1)
        den = W.sum(axis=0, keepdims=True)
    return W / den


def adopt_config_globals(cfg):
    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM, SLICE_BAND, RULES
    CACHE_IMG = IMG = int(cfg["img"])
    GROUP = int(cfg["group"])
    CACHE_SLICES = int(cfg["slices"])
    N_GROUP = max(CACHE_SLICES // GROUP, 1)
    CROP_MM = float(cfg["crop_mm"])
    SLICE_BAND = tuple(float(x) for x in cfg["band"])
    rules = cfg.get("rules") or RULES_NATIVE
    unknown = {k: v for k, v in rules.items()
               if k not in RULES_NATIVE or v not in (RULES_NATIVE[k], RULES_LEGACY[k])}
    if unknown:
        raise WeightsError(f"unsupported pixel rules in manifest: {unknown}")
    RULES = {**RULES_NATIVE, **rules}
    if [s[0] for s in SLOTS] != list(cfg["slots"]):
        raise WeightsError(
            f"manifest slots {cfg['slots']} != notebook slots {[s[0] for s in SLOTS]}")


def write_submission(pred, studies, test_df, path):
    ranked = _rank_matrix(pred)
    sub = pd.DataFrame(ranked, columns=TARGETS)
    sub.insert(0, "StudyInstanceUID", studies)
    sub = test_df[["StudyInstanceUID"]].merge(sub, on="StudyInstanceUID", how="left")
    sub[TARGETS] = sub[TARGETS].fillna(0.5)
    sub.to_csv(path, index=False)
    return sub


def write_benchmark_submission():
    t = pd.read_csv(ROOT / "test.csv")
    for c in TARGETS:
        t[c] = 0.5
    t.to_csv("submission.csv", index=False)


def infer_from_package_enhanced(path, dev):
    man = json.loads((Path(path) / "manifest.json").read_text())
    members = man["members"]
    log(f"weights package: {len(members)} member(s) from {path}")

    test_df = pd.read_csv(ROOT / "test.csv")
    test_series = pd.read_csv(ROOT / "test_series.csv")
    plane_map = dict(zip(test_series["SeriesInstanceUID"], test_series["Anatomical_Plane"]))
    hte = annotate(walk("test_series"))
    log(f"test header pass: {len(hte)} series")
    audit_official_sequence_metadata(hte, test_series)

    groups = {}
    for m in members:
        groups.setdefault(m["pixel_group"], []).append(m)

    per_member = []
    fixed_s = per_win_s = None

    group_items = list(groups.items())
    budget_exhausted = False
    for gi, (key, gm) in enumerate(group_items, 1):
        if budget_exhausted:
            log(f"  skipping group {gi}/{len(groups)}: time budget already exhausted")
            break

        cfg = json.loads(key)
        adopt_config_globals(cfg)
        log(f"decode group {gi}/{len(groups)}: {cfg['img']}px x {cfg['slices']} slices, "
            f"crop {cfg['crop_mm']} mm -> {len(gm)} member(s)")

        st_te, Cte, Mte = build_cache(
            pick_slots(hte, plane_map), plane_map, lat_of(hte, "test "), f"test g{gi}")
        idx = np.arange(len(st_te))
        starts = window_starts(Cte.shape[2], GROUP)
        order = sorted(gm, key=lambda m: -(m.get("holdout") or 0))
        left_after = sum(len(g) for _, g in group_items[gi:])

        for k, m in enumerate(order):
            left = TIME_BUDGET - (time.time() - T0)
            remaining = (len(order) - k) + left_after
            use_starts = list(starts)

            if fixed_s is not None and per_win_s is not None:
                afford = max(left * 0.9, 0.0)
                need = fixed_s + len(use_starts) * per_win_s
                if need * remaining > afford:
                    room = afford / max(remaining, 1)
                    n_win = int((room - fixed_s) / per_win_s) if per_win_s > 0 else 0
                    n_win = max(1, min(len(use_starts), n_win))
                    if fixed_s + per_win_s > afford:
                        log(f"  {left / 60:.0f} min left: stopping; one more member does not fit")
                        # Stopping here only breaks this group's member loop. Without also
                        # marking the budget exhausted, the outer loop would still decode
                        # and attempt the next pixel-group from scratch - exactly the
                        # expensive step this guard exists to avoid once time has run out.
                        budget_exhausted = True
                        break
                    if n_win < len(use_starts):
                        mid = (len(use_starts) - n_win) // 2
                        use_starts = use_starts[mid:mid + n_win]
                        log(f"  {left / 60:.0f} min left: using {n_win}/{len(starts)} TTA windows")

            t0 = time.time()
            ck = torch.load(Path(path) / m["file"], map_location="cpu", weights_only=False)
            model = build_model(
                int(m["config"]["unfreeze_last"]),
                variant=m["config"]["variant"],
                pool=m["config"].get("pool", "cls_mean"),
                prior=bool(m["config"].get("prior", False)),
            ).to(dev)
            model.load_state_dict(ck["model"])
            check_fingerprint(model, dev, IMG, ck["fingerprint"], tag=f"{m['id']}: ")
            target_auc = extract_target_auc(m, ck)
            t_ready = time.time()

            p, views = predict_member_views(
                model, Cte, Mte, idx, dev, IMG, starts=use_starts)
            per_member.append({
                "id": m["id"],
                "fold": m.get("fold"),
                "ids": st_te,
                "pred": p,
                "views": views,
                "holdout": m.get("holdout"),
                "target_auc": target_auc,
            })

            fixed_s = t_ready - t0
            per_win_s = (time.time() - t_ready) / max(len(use_starts), 1)
            log(f"  {m['id']} fold {m.get('fold')}: {len(idx)} studies, "
                f"{len(use_starts)} windows, {time.time() - t0:.0f}s")
            del model, ck
            gc.collect()
            if dev.type == "cuda":
                torch.cuda.empty_cache()

        del Cte, Mte
        gc.collect()

    if not per_member:
        raise WeightsError("no member produced predictions")

    all_ids = sorted({s for m in per_member for s in m["ids"]})
    pos = {s: i for i, s in enumerate(all_ids)}
    M, N, T = len(per_member), len(all_ids), len(TARGETS)
    member_rank = np.full((M, N, T), np.nan, np.float64)
    window_rank = np.full((M, N, T), np.nan, np.float64)

    for mi, m in enumerate(per_member):
        rows = [pos[s] for s in m["ids"]]
        member_rank[mi, rows] = _rank_matrix(m["pred"])
        wr = np.mean([_rank_matrix(v) for v in m["views"]], axis=0)
        window_rank[mi, rows] = wr

    if np.isnan(member_rank).any():
        raise WeightsError("some members did not cover all test studies")

    baseline = member_rank.mean(axis=0)  # exact public 0.891-style equal member rank mean
    window_vote = window_rank.mean(axis=0)

    W_scalar = fold_balanced_member_weights(per_member, target_specific=False)
    quality = np.einsum("mnt,mt->nt", member_rank, W_scalar)

    has_target_auc = any(m.get("target_auc") is not None for m in per_member)
    if has_target_auc:
        W_target = fold_balanced_member_weights(per_member, target_specific=True)
        target_quality = np.einsum("mnt,mt->nt", member_rank, W_target)
        hybrid = (HYBRID_TARGET_BASE * baseline +
                  HYBRID_TARGET_QUALITY * target_quality +
                  HYBRID_TARGET_WINDOW * window_vote)
        log("hybrid uses per-target validation metadata + window ranks")
    else:
        target_quality = quality
        hybrid = (HYBRID_BASE * baseline +
                  HYBRID_QUALITY * quality +
                  HYBRID_WINDOW * window_vote)
        log("no per-target AUC metadata found; hybrid uses within-fold scalar holdout + window ranks")

    # Always write alternatives. This is deliberate: no inference-only change can be
    # guaranteed to beat an unseen leaderboard, so the proven public branch is preserved.
    write_submission(baseline, all_ids, test_df, "submission_baseline_0891.csv")
    write_submission(quality, all_ids, test_df, "submission_foldquality.csv")
    write_submission(window_vote, all_ids, test_df, "submission_windowrank.csv")
    write_submission(hybrid, all_ids, test_df, "submission_hybrid.csv")

    choices = {
        "baseline": baseline,
        "quality": quality,
        "window": window_vote,
        "hybrid": hybrid,
    }
    chosen_name = ENSEMBLE_MODE if ENSEMBLE_MODE in choices else "hybrid"
    chosen = choices[chosen_name]
    sub = write_submission(chosen, all_ids, test_df, "submission.csv")

    diag = pd.DataFrame({
        "member": [m["id"] for m in per_member],
        "fold": [m.get("fold") for m in per_member],
        "holdout": [m.get("holdout") for m in per_member],
        "n_windows": [len(m["views"]) for m in per_member],
        "has_target_auc": [m.get("target_auc") is not None for m in per_member],
    })
    display(diag.sort_values(["fold", "holdout"], ascending=[True, False]))
    log(f"submission.csv mode={chosen_name}; {len(per_member)} member(s); {sub.shape}; "
        f"nulls={int(sub[TARGETS].isna().sum().sum())}")
    log("also wrote: submission_baseline_0891.csv, submission_foldquality.csv, "
        "submission_windowrank.csv, submission_hybrid.csv")
    return sub


def require_cuda():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required for this final inference notebook")
    return torch.device("cuda")


def main():
    write_benchmark_submission()
    pkg = find_weights()
    if pkg is None:
        raise FileNotFoundError(
            "Attach the same weights dataset used by the 0.891 notebook. Expected first at "
            "/kaggle/input/rsna-knee-weights (or another mounted package with manifest.json).")
    return infer_from_package_enhanced(pkg, require_cuda())


## 10. Run inference and write submissions

In [ ]:
# ── Manifest / checkpoint sanity check (run this BEFORE the weights are loaded) ──
# Scans /kaggle/input for your attached weights dataset, validates manifest.json
# against exactly what infer_from_package_enhanced() expects, and STOPS the run
# with a clear error instead of failing hours into inference.

import json
from pathlib import Path
import torch

SLOTS_RECOVERED = ["SAG_FLUID_FS", "COR_FLUID_FS", "AX_FLUID_FS",
                    "SAG_FLUID_NOFS", "COR_T1", "SAG_T1"]
SLOTS_PUBLIC = ["SAG_FLUID", "COR_FLUID", "AX_FLUID",
                "SAG_STRUCT", "COR_STRUCT", "AX_STRUCT"]
RULES_NATIVE = {"order": "normal", "lat": "centre",
                "slot_fallback": False, "decode_fill": "nearest"}
RULES_LEGACY = {"order": "dominant_axis", "lat": "corner_x",
                 "slot_fallback": True, "decode_fill": "zero"}


def _find_manifest_dir():
    for p in Path("/kaggle/input").glob("*/manifest.json"):
        return p.parent
    for p in Path("/kaggle/input").glob("*/*/manifest.json"):
        return p.parent
    return None


root = _find_manifest_dir()
if root is None:
    raise FileNotFoundError(
        "No manifest.json found anywhere under /kaggle/input -- "
        "attach your rsna-knee-weights dataset via Add Input first."
    )
print(f"checking weights dataset at: {root}")

man = json.loads((root / "manifest.json").read_text())
members = man.get("members")
assert isinstance(members, list) and members, "manifest.json has no 'members' list"
print(f"{len(members)} member(s) listed\n")

errors = []
for i, m in enumerate(members):
    mid = m.get("id")
    tag = f"member[{i}] (id={mid!r})"

    if not mid:
        errors.append(f"{tag}: missing id")
    if not m.get("file"):
        errors.append(f"{tag}: missing file")
        continue
    ck_path = root / str(m["file"])
    if not ck_path.is_file():
        errors.append(f"{tag}: checkpoint file not found: {ck_path.name}")
        continue

    cfg = m.get("config")
    if not isinstance(cfg, dict):
        errors.append(f"{tag}: missing config dict")
    else:
        for key in ("unfreeze_last", "variant"):
            if key not in cfg:
                errors.append(f"{tag}: config missing key '{key}'")

    pg = m.get("pixel_group")
    if not pg:
        errors.append(f"{tag}: missing pixel_group")
    else:
        pcfg = json.loads(pg) if isinstance(pg, str) else pg
        for key in ("img", "group", "slices", "crop_mm", "band", "slots"):
            if key not in pcfg:
                errors.append(f"{tag}: pixel_group missing key '{key}'")
        slots = pcfg.get("slots")
        if slots not in (SLOTS_RECOVERED, SLOTS_PUBLIC):
            errors.append(
                f"{tag}: pixel_group.slots={slots} does not match "
                f"SLOTS_RECOVERED/SLOTS_PUBLIC -- adopt_config_globals() "
                f"will raise WeightsError"
            )
        rules = pcfg.get("rules") or {}
        bad_rules = {
            k: v for k, v in rules.items()
            if k not in RULES_NATIVE or v not in (RULES_NATIVE[k], RULES_LEGACY[k])
        }
        if bad_rules:
            errors.append(f"{tag}: pixel_group.rules has unsupported entries: {bad_rules}")

    try:
        ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    except Exception as e:
        errors.append(f"{tag}: torch.load failed: {e}")
        continue
    if "model" not in ck:
        errors.append(f"{tag}: checkpoint has no 'model' key")
    if "fingerprint" not in ck:
        errors.append(f"{tag}: checkpoint has no 'fingerprint' key")

    print(f"  [ok] {tag}: {ck_path.name}")

if errors:
    msg = "\n".join(errors)
    raise AssertionError(f"\n{len(errors)} problem(s) in manifest.json:\n{msg}")

print(f"\nAll {len(members)} member(s) look structurally correct -- safe to continue.")


In [ ]:
try:
    submission = main()
    display(submission.head())
    log("done")
except Exception:
    traceback.print_exc()
    raise



### Output files

- `submission.csv` — selected by `RSNA_ENSEMBLE_MODE` (default: `hybrid`)
- `submission_baseline_0891.csv` — exact equal member-rank baseline
- `submission_foldquality.csv` — fold-balanced, shrinked quality ranks
- `submission_windowrank.csv` — TTA-window rank vote
- `submission_hybrid.csv` — conservative blend used by default

**Why the old ~0.83 checkpoint is not auto-blended:** a weaker model can improve an ensemble only when its errors are complementary. Without OOF predictions proving that, blindly mixing it into a 0.891 ensemble is more likely to reduce the score. The old notebook's useful contribution here is the EDA/metadata audit, not an unvalidated extra vote.
